# `code/pipeline/p001_49_coattribution_audit.py`

Read-only rendering of the script (no outputs; it needs the licensed inputs described in `../../DATA_ACCESS.md`). The .py file is the version of record.


```text
p001_49 — 동료 비교의 정체: 혼합 셀의 대부분이 **같은 라운드의 공동귀속 쌍**이다 — 결과 분산이 있는 셀만으로 본 헤드라인

[왜] P001-48 의 유럽 판이 β = 0.00 [0, 0] 을 냈다. 진단(2026-09-09): 유럽 혼합 셀의 85–93% 가 **한 라운드**만 담는다 — 여성·남성 파트너가
 같은 딜에 공동 귀속된 쌍이라 결과(exit)가 셀 안에서 상수 → 식별 변이 0. 미국도 cell_cat 혼합 셀의 66%, cell_stage 의 87% 가 그렇고, 셀 내
 exit 분산이 있는 셀은 17% / 6% 뿐이다. 즉 Table 3 의 셀 내 추정량은 (a) 결과 분산이 있는 소수의 셀이 분자를 만들고 (b) 공동귀속 쌍은
 분모(Σxr²)만 키워 **계수를 0 쪽으로 희석**한다. 원고가 서술하는 "같은 회사·해·섹터의 동료 딜과의 비교" 는 (a) 만이다.

[구성] P001-10 사양(FF 딜, exit_ever, ≤2017-10; cell_cat / cell_stage; 투자사 군집 부트 500) — GLOBAL·NAEU.
 셀 분류: 단일 라운드(공동귀속만) / 다중 라운드·결과 분산>0 / 다중 라운드·결과 분산=0.
 A 분류 카운트 · Σxr² 의 단일 라운드 셀 비중(희석 비중) · β 전체(재현) · **β 다중 라운드 셀만**(다른 딜 간 동료 비교) · 그 CI · 셀 내 재배정 위약 p95
 B 같은 것을 fon(후속) 에도.
[사전 예측] (2026-09-09, 결과 조회 전)
 NAEU cell_cat: 단일 라운드 셀 비중 0.60–0.70; 결과 분산>0 셀 25–40; β_다중 ∈ [−12, −5]pp (희석 제거로 커짐), CI 반폭 ≥ 8; 위약 p95 9–13.
 GLOBAL cell_cat: 결과 분산>0 셀 35–55; β_다중 ∈ [−12, −5]. cell_stage 는 셀 ≤ 15 로 사실상 추정 불가.
[판정] 감사 — status OK. 원고 §4 에 "동료 비교의 실제 식별 기반" 으로 서술.
```


In [ ]:
import numpy as np
import pandas as pd

from p001_rescue_common import COMMON_SHA, CUT, NAEU, emit, load_deals, log, qci

rng = np.random.default_rng(20260949)
NB, NPL = 500, 400
OUT = {}
dn_all = load_deals.__globals__["pd"].read_parquet(__import__("os").environ.get("P001_SAMPLE", "/path/to/sample_v1.parquet"))
dn_all["dt"] = pd.to_datetime(dn_all["dt"])
ffd_all = dn_all[(dn_all["ff"] == 1) & (dn_all["dt"] <= CUT)].copy()


def within_beta(dd, cell, y):
    yr = (dd[y] - dd[y].groupby(dd[cell]).transform("mean")).to_numpy()
    xr = (dd["fp"] - dd["fp"].groupby(dd[cell]).transform("mean")).to_numpy()
    sxx = float((xr * xr).sum())
    return float((xr * yr).sum() / sxx) if sxx > 0 else np.nan


def boot_ci(dd, cell, y, nb=NB):
    grp = {c: g.index.to_numpy() for c, g in dd.groupby("investor_uuid")}; kl = list(grp); bs = []
    for _ in range(nb):
        pick = rng.integers(0, len(kl), len(kl)); s = dd.loc[np.concatenate([grp[kl[i]] for i in pick])]
        v = within_beta(s, cell, y)
        if np.isfinite(v): bs.append(v)
    lo, hi = qci(np.array(bs)); return [round(lo * 100, 2), round(hi * 100, 2)], round(2.8 * float(np.std(bs, ddof=1)) * 100, 2)


def placebo_p95(m, cell, y, n=NPL):
    mm = m.reset_index(drop=True); yr = (mm[y] - mm[y].groupby(mm[cell]).transform("mean")).to_numpy(); pl = []
    for _ in range(n):
        fpp = mm.groupby(cell)["fp"].transform(lambda s: pd.Series(rng.permutation(s.to_numpy()), index=s.index))
        xr = (fpp - fpp.groupby(mm[cell]).transform("mean")).to_numpy(); sxx = float((xr * xr).sum())
        if sxx > 0: pl.append(abs(float((xr * yr).sum() / sxx)) * 100)
    return round(float(np.percentile(pl, 95)), 2) if pl else None


for scope in ("GLOBAL", "NAEU"):
    ffd = (ffd_all if scope == "GLOBAL" else ffd_all[ffd_all["country_code"].isin(NAEU)]).reset_index(drop=True)
    res = {"n_ff_deals": int(len(ffd))}
    log("\n" + "=" * 100 + f"\n[{scope}] FF 딜 {len(ffd):,}\n" + "=" * 100)
    for cell in ("cell_cat", "cell_stage"):
        g = ffd.groupby(cell)["fp"].agg(["mean", "size"]); mixed = g.index[(g["mean"] > 0) & (g["mean"] < 1)]
        m = ffd[ffd[cell].isin(mixed)].copy()
        nr = m.groupby(cell)["funding_round_uuid"].nunique()
        single = set(nr.index[nr == 1]); multi = set(nr.index[nr >= 2])
        yv = m.groupby(cell)["exit_ever"].var(ddof=0).fillna(0); varpos = set(yv.index[yv > 0])
        xr = (m["fp"] - m.groupby(cell)["fp"].transform("mean")) ** 2
        dil = float(xr[m[cell].isin(single)].sum() / xr.sum())
        r = {"n_mixed_cells": int(len(mixed)), "n_deals_mixed": int(len(m)), "n_single_round_cells": len(single), "share_single_round_cells": round(len(single) / max(len(mixed), 1), 4),
             "n_multi_round_cells": len(multi), "n_deals_multi": int(m[cell].isin(multi).sum()), "n_cells_exit_var_pos": len(varpos), "n_deals_exit_var_pos": int(m[cell].isin(varpos).sum()),
             "dilution_share_sxx_single_round": round(dil, 4), "beta_full_pp": round(within_beta(ffd, cell, "exit_ever") * 100, 3)}
        mm = m[m[cell].isin(multi)]
        for y in ("exit_ever", "fon"):
            if len(mm) >= 30 and mm[cell].nunique() >= 8:
                b = within_beta(mm, cell, y); ci, mde = boot_ci(mm, cell, y)
                r[f"beta_multi_{y}_pp"] = round(b * 100, 3); r[f"ci_multi_{y}_pp"] = ci; r[f"mde_multi_{y}_pp"] = mde; r[f"placebo_p95_multi_{y}_pp"] = placebo_p95(mm, cell, y)
            else:
                r[f"beta_multi_{y}_pp"] = None
        res[cell] = r
        log(f"  {cell:<10} 혼합 {r['n_mixed_cells']} 셀/{r['n_deals_mixed']} 딜 · 단일라운드 셀 {r['n_single_round_cells']} ({r['share_single_round_cells']:.2f}) · 다중 {r['n_multi_round_cells']} 셀/{r['n_deals_multi']} 딜 · exit 분산>0 {r['n_cells_exit_var_pos']} 셀/{r['n_deals_exit_var_pos']} 딜 · Σxr² 희석 {r['dilution_share_sxx_single_round']:.2f} · "
            f"β 전체 {r['beta_full_pp']:+.2f} · β 다중 exit {r.get('beta_multi_exit_ever_pp')} {r.get('ci_multi_exit_ever_pp')} MDE {r.get('mde_multi_exit_ever_pp')} 위약 p95 {r.get('placebo_p95_multi_exit_ever_pp')} · fon {r.get('beta_multi_fon_pp')} {r.get('ci_multi_fon_pp')}")
    OUT[scope] = res

n = OUT["NAEU"]["cell_cat"]; g_ = OUT["GLOBAL"]["cell_cat"]
pred = {"NAEU_single_share_0.60_0.70": 0.60 <= n["share_single_round_cells"] <= 0.70, "NAEU_varpos_25_40": 25 <= n["n_cells_exit_var_pos"] <= 40,
        "NAEU_beta_multi_in_[-12,-5]": bool(n.get("beta_multi_exit_ever_pp") is not None and -12 <= n["beta_multi_exit_ever_pp"] <= -5),
        "NAEU_halfwidth_ge_8": bool(n.get("ci_multi_exit_ever_pp") and (n["ci_multi_exit_ever_pp"][1] - n["ci_multi_exit_ever_pp"][0]) / 2 >= 8),
        "GLOBAL_varpos_35_55": 35 <= g_["n_cells_exit_var_pos"] <= 55, "GLOBAL_beta_multi_in_[-12,-5]": bool(g_.get("beta_multi_exit_ever_pp") is not None and -12 <= g_["beta_multi_exit_ever_pp"] <= -5)}
pred = {k: bool(v) for k, v in pred.items()}
OUT["prediction_check"] = pred
verdict = (f"NAEU cat: 혼합 {n['n_mixed_cells']} 셀 중 단일라운드(공동귀속) {n['n_single_round_cells']} ({n['share_single_round_cells']:.0%}), exit 분산>0 셀 {n['n_cells_exit_var_pos']}/{n['n_deals_exit_var_pos']} 딜, Σxr² 희석 {n['dilution_share_sxx_single_round']:.0%}; "
           f"β 전체 {n['beta_full_pp']:+.2f} → 다중라운드 셀만 {n.get('beta_multi_exit_ever_pp')} {n.get('ci_multi_exit_ever_pp')} (MDE {n.get('mde_multi_exit_ever_pp')}, 위약 p95 {n.get('placebo_p95_multi_exit_ever_pp')}) | "
           f"GLOBAL cat: 단일 {g_['share_single_round_cells']:.0%}, 분산>0 {g_['n_cells_exit_var_pos']} 셀; β 전체 {g_['beta_full_pp']:+.2f} → 다중 {g_.get('beta_multi_exit_ever_pp')} {g_.get('ci_multi_exit_ever_pp')} | "
           f"NAEU stage: 단일 {OUT['NAEU']['cell_stage']['share_single_round_cells']:.0%}, 분산>0 {OUT['NAEU']['cell_stage']['n_cells_exit_var_pos']} 셀 (예측 적중 {sum(pred.values())}/{len(pred)})")
emit("P001-49", "동료 비교의 식별 기반: 공동귀속(단일 라운드) 셀 비중 · 결과 분산이 있는 셀 · 희석 · 다중 라운드 셀만의 격차", "OK", OUT,
     prediction="NAEU cat 단일라운드 0.60–0.70; 분산>0 셀 25–40; β_다중 ∈[−12,−5] 반폭≥8; GLOBAL 분산>0 35–55",
     verdict=verdict, kill_met=False, n=int(OUT["NAEU"]["n_ff_deals"]),
     extra={"stage": 7, "feeds": "§4 동료 비교 서술 · Table 9 (P001-40 보완)", "slug": "coattribution_audit", "builds_on": "P001-10/40/48", "common_sha256_16": COMMON_SHA})
log("done")
